# Local Eval Harness Demo

This notebook drives `adk.eval_harness.local_harness` — `run_and_score()` and `rollup()` — against
`adk.demos.plan_then_act_demo.DemoPlanThenActAgent`, the same live agent walked through in
[`plan_then_act_demo.ipynb`](plan_then_act_demo.ipynb).

`local_harness` is deliberately small: no `langsmith.Client`, no network calls beyond the agent
itself. It takes a list of `EvalCase`s, invokes the agent on each one, scores the result with a
list of pure metric functions from `eval_harness.metrics`, and rolls the scored rows up into
summary numbers. (`eval_harness.harness` — `SingleLLMEvalHarness` — is a separate, unrelated,
LangSmith-backed harness; this notebook doesn't touch it.)

The agent and harness code both live in `src/adk/` — this notebook imports them and walks
through what they do, rather than redefining any of the logic here.

## Setup

Requires two API keys:

- `ANTHROPIC_API_KEY` — https://console.anthropic.com/
- `TAVILY_API_KEY` — https://tavily.com/

Copy `.env.example` (repo root) to `.env` and paste your keys in there — `load_dotenv()`
below loads it into this process. `.env` is gitignored, so real keys never get committed.

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

for key in ("ANTHROPIC_API_KEY", "TAVILY_API_KEY"):
    print(f"{key}: {'set' if os.environ.get(key) else 'MISSING'}")

ANTHROPIC_API_KEY: set
TAVILY_API_KEY: set


## `EvalCase` and `RunResult`

`eval_harness.cases.EvalCase` is one dataset row: an `id`, a `task` string, optional `context`,
and optional ground truth — `expected_output` and/or `expected_steps` (an ordered list of
`ExpectedStep(executor_id, tool_name)` the run *should* take).

`run_and_score` doesn't need a `RunResult` from us — it invokes the agent itself and builds one
via `PlannerExecutorBase.to_eval_run_result()`, which already knows how to pull `executed_steps`
and trace metadata (`run_id`, `latency_ms`, `tokens_in`/`tokens_out`) out of the agent's raw
`invoke()` output.

In [2]:
from adk.demos.plan_then_act_demo import DEFAULT_TASK, DemoPlanThenActAgent
from adk.eval_harness.cases import EvalCase, ExpectedStep
from adk.eval_harness.local_harness import rollup, run_and_score
from adk.eval_harness.metrics import (
    latency,
    plan_execution_alignment,
    steps_to_completion,
    task_success,
    token_counts,
)

## Building the agent

Same agent as the plan-then-act demo — nothing eval-specific about its construction. It owns two
executors (`search` -> `web_search`, `calc` -> `calculate`); the eval cases below are chosen to
exercise both.

In [3]:
agent = DemoPlanThenActAgent()

## Defining eval cases

Three cases, each with `expected_steps` — the routing the planner *should* choose:

- **`population_lookup`** — a pure lookup. Should route to `search` only; routing it to `calc`
  instead (a wrong-tool mistake `plan_execution_alignment` is built to catch) would be a bug.
- **`arithmetic`** — pure arithmetic, no lookup needed. Should route to `calc` only.
- **`combined_population`** — reuses the plan-then-act demo's own `DEFAULT_TASK`. This is the
  case with real stakes for `plan_execution_alignment`: it needs *two* `search` steps (France,
  Germany) followed by one `calc` step that depends on both — a genuine multi-step plan across
  two heterogeneous executors, not just a structural check that execution obeyed its own plan
  (the execution coordinator already guarantees that part).

In [4]:
CASES = [
    EvalCase(
        id="population_lookup",
        task="What is the current population of France?",
        expected_steps=[ExpectedStep(executor_id="search", tool_name="web_search")],
    ),
    EvalCase(
        id="arithmetic",
        task="What is 482 times 17?",
        expected_steps=[ExpectedStep(executor_id="calc", tool_name="calculate")],
    ),
    EvalCase(
        id="combined_population",
        task=DEFAULT_TASK,
        expected_steps=[
            ExpectedStep(executor_id="search", tool_name="web_search"),
            ExpectedStep(executor_id="search", tool_name="web_search"),
            ExpectedStep(executor_id="calc", tool_name="calculate"),
        ],
    ),
]

## Choosing metrics

Each metric in `eval_harness.metrics` is a pure `(EvalCase, RunResult) -> dict` function — no
shared "result" envelope, no side effects. This demo runs five of them:

- **`task_success`** — heuristic pass/fail: did the run complete with zero `"degraded"` steps?
- **`steps_to_completion`** — how many steps the run actually executed.
- **`plan_execution_alignment`** — do the executed steps match `expected_steps`, 1:1, in order?
  This is the one that catches genuine planner mistakes, e.g. routing a lookup to `calculate`.
- **`latency`** — wall-clock time for the whole `invoke()` call, in ms.
- **`token_counts`** — input/output/total tokens across every `AnthropicRunnable` call in the run.

(`plan_adequacy` also exists in `metrics.py`, but it's a stub for a not-yet-wired LLM judge —
left out here since it has nothing to show yet.)

In [5]:
METRICS = [task_success, steps_to_completion, plan_execution_alignment, latency, token_counts]

## Running the harness

`run_and_score` invokes the agent on each case in turn and scores it — three real
Anthropic + Tavily round trips, so this cell takes a little while.

The planner is a live LLM call behind a forced tool call; on rare occasions it used to emit
the plan's `steps` field double-encoded as a JSON string instead of a native array, which
`_parse_plan_artifact` (`planner_executor/planner.py`) now detects and decodes before
validating — so this cell shouldn't need a retry, but the coercion only handles that one known
shape.

In [6]:
rows = run_and_score(CASES, agent, METRICS)

## Scored rows

One flat dict per case — `case_id`, `run_id`, and every metric's fields merged together
(built-in metrics use disjoint keys, so nothing collides).

In [7]:
for row in rows:
    print(f"{row['case_id']!r}:")
    for key, value in row.items():
        if key in ("case_id",):
            continue
        print(f"  {key}: {value}")

'population_lookup':
  run_id: 8864712b-d049-4c77-abcf-1004c90e6022
  success: True
  n_steps: 1
  n_degraded_steps: 0
  degraded_step_indices: []
  steps_to_completion: 1
  aligned: True
  n_expected_steps: 1
  n_executed_steps: 1
  mismatches: []
  latency_ms: 7285.102333000395
  tokens_in: 3308
  tokens_out: 279
  tokens_total: 3587
'arithmetic':
  run_id: 294c0e60-e964-4c1e-8c93-0ad7b496359d
  success: True
  n_steps: 1
  n_degraded_steps: 0
  degraded_step_indices: []
  steps_to_completion: 1
  aligned: True
  n_expected_steps: 1
  n_executed_steps: 1
  mismatches: []
  latency_ms: 2887.7162919961847
  tokens_in: 1434
  tokens_out: 91
  tokens_total: 1525
'combined_population':
  run_id: 4eb2e656-6e56-401e-a094-2f71b897cd82
  success: True
  n_steps: 3
  n_degraded_steps: 0
  degraded_step_indices: []
  steps_to_completion: 3
  aligned: True
  n_expected_steps: 3
  n_executed_steps: 3
  mismatches: []
  latency_ms: 7765.869457973167
  tokens_in: 5561
  tokens_out: 458
  tokens_tot

## Rollup

`rollup()` aggregates the scored rows into three numbers — pass rate, average steps to
completion, and alignment rate — each computed only over the rows that carry the relevant
metric's key, so it degrades gracefully if a case was scored with a partial metric list.

In [8]:
rollup(rows)

{'n_cases': 3,
 'pass_rate': 1.0,
 'avg_steps_to_completion': 1.6666666666666667,
 'alignment_rate': 1.0}

## What `plan_execution_alignment` actually catches

`aligned=True` for a row means the executed steps matched `expected_steps` exactly, in order.
When it's `False`, the row's `mismatches` list pinpoints exactly which step index disagreed and
how — this is the signal that would flag, for example, a lookup step the planner routed to
`calculate` instead of `web_search`.

In [9]:
for row in rows:
    if not row["aligned"]:
        print(f"{row['case_id']}: {row['mismatches']}")
    else:
        print(f"{row['case_id']}: aligned")

population_lookup: aligned
arithmetic: aligned
combined_population: aligned


## Scoring the generated eval cases

`adk.demos.generate_eval_cases_demo` (`uv run python -m adk.demos.generate_eval_cases_demo`) drives the generate-evaluate-reflect closed-loop graph to synthesize additional cases in the same three categories as `CASES` above, and writes them to [`data/generated_eval_cases.json`](data/generated_eval_cases.json). This section loads that file with `eval_harness.cases.load_eval_cases` and scores it through the same harness and metrics, so the generated set can be compared directly against the hand-written rollup above.

In [10]:
from adk.eval_harness.cases import load_eval_cases

GENERATED_CASES = load_eval_cases("data/generated_eval_cases.json")
len(GENERATED_CASES)

6

In [11]:
generated_rows = run_and_score(GENERATED_CASES, agent, METRICS)

for row in generated_rows:
    print(f"{row['case_id']!r}:")
    for key, value in row.items():
        if key in ("case_id",):
            continue
        print(f"  {key}: {value}")

'generated_search_only_1':
  run_id: 3039456c-076b-44f4-ba42-4747be3cd357
  success: True
  n_steps: 1
  n_degraded_steps: 0
  degraded_step_indices: []
  steps_to_completion: 1
  aligned: True
  n_expected_steps: 1
  n_executed_steps: 1
  mismatches: []
  latency_ms: 4992.516375030391
  tokens_in: 2430
  tokens_out: 191
  tokens_total: 2621
'generated_search_only_2':
  run_id: 6fd5374a-d983-463d-be30-b7178eddfcb9
  success: True
  n_steps: 1
  n_degraded_steps: 0
  degraded_step_indices: []
  steps_to_completion: 1
  aligned: True
  n_expected_steps: 1
  n_executed_steps: 1
  mismatches: []
  latency_ms: 5717.833917005919
  tokens_in: 4119
  tokens_out: 260
  tokens_total: 4379
'generated_calc_only_1':
  run_id: 18200183-c09a-45db-b260-3b6690c6a73f
  success: True
  n_steps: 1
  n_degraded_steps: 0
  degraded_step_indices: []
  steps_to_completion: 1
  aligned: True
  n_expected_steps: 1
  n_executed_steps: 1
  mismatches: []
  latency_ms: 2732.0393749978393
  tokens_in: 1439
  tokens

Rollup for the generated set, alongside the hand-written one computed earlier:

In [12]:
rollup(generated_rows)

{'n_cases': 6,
 'pass_rate': 1.0,
 'avg_steps_to_completion': 1.6666666666666667,
 'alignment_rate': 1.0}